In [1]:
!pip install pandas scikit-image opencv-python

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from skimage.transform import resize

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

2026-03-07 14:47:43.878099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 14:47:43.908139: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 14:47:43.908184: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 14:47:43.927961: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 14:47:44.925252: W tensorflow/com

In [3]:
DATASET_PATH = "dataset/Dataset for Fetus Framework"

TRAIN_STD = DATASET_PATH + "/Set1-Training&Validation Sets CNN/Standard"
TRAIN_ANN_STD = DATASET_PATH + "/Set2-Training&Validation Sets ANN Scoring system/Standard"
TRAIN_ANN_NON = DATASET_PATH + "/Set2-Training&Validation Sets ANN Scoring system/Non-standard"

INTERNAL_TEST = DATASET_PATH + "/Internal Test Set"
EXTERNAL_TEST = DATASET_PATH + "/External Test Set"

EXCEL_FILE = "dataset/ObjectDetection.xlsx"

In [4]:
df = pd.read_excel(EXCEL_FILE)

# keep only NT rows
nt_df = df[df["structure"] == "NT"]

nt_df.head()

,fname,structure,h_min,w_min,h_max,w_max
6,168.png,NT,385,257,418,464
16,169.png,NT,372,126,428,371
20,170.png,NT,376,12,416,478
30,171.png,NT,359,173,389,529
37,172.png,NT,375,293,399,380


In [5]:
def get_image_paths(folder):

    paths = []

    if not os.path.exists(folder):
        return paths

    for file in os.listdir(folder):

        if file.endswith(".png"):
            paths.append(os.path.join(folder,file))

    return paths


paths1 = get_image_paths(TRAIN_STD)
paths2 = get_image_paths(TRAIN_ANN_STD)
paths3 = get_image_paths(TRAIN_ANN_NON)

all_paths = paths1 + paths2 + paths3

print("Total images:",len(all_paths))

Total images: 1368


In [6]:
train_paths, val_paths = train_test_split(
    all_paths,
    test_size=0.2,
    random_state=42
)

print(len(train_paths),len(val_paths))

1094 274


In [7]:
def load_image_mask(path):

    file = os.path.basename(path)

    img = cv2.imread(path)

    h_original,w_original = img.shape[:2]

    img = cv2.resize(img,(256,256))
    img = img / 255.0

    mask = np.zeros((256,256,1),dtype=np.float32)

    row = nt_df[nt_df["fname"]==file]

    if len(row)>0:

        h_min = int(row.iloc[0]["h_min"])
        w_min = int(row.iloc[0]["w_min"])
        h_max = int(row.iloc[0]["h_max"])
        w_max = int(row.iloc[0]["w_max"])

        h_scale = 256/h_original
        w_scale = 256/w_original

        h_min = int(h_min*h_scale)
        h_max = int(h_max*h_scale)
        w_min = int(w_min*w_scale)
        w_max = int(w_max*w_scale)

        mask[h_min:h_max,w_min:w_max] = 1

    return img.astype(np.float32),mask

In [8]:
def parse_image(path):

    path = path.numpy().decode()

    img,mask = load_image_mask(path)

    return img,mask


def tf_parse(path):

    img,mask = tf.py_function(
        parse_image,
        [path],
        [tf.float32,tf.float32]
    )

    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])

    return img,mask

In [9]:
def augment(image,mask):

    if tf.random.uniform(())>0.5:

        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    if tf.random.uniform(())>0.5:

        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    return image,mask

In [12]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model("best_model.keras", compile=False)

In [13]:
BATCH = 8

test_dataset = tf.data.Dataset.from_tensor_slices(val_paths)
test_dataset = test_dataset.map(tf_parse)
test_dataset = test_dataset.batch(BATCH)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [14]:
pred_masks = model.predict(test_dataset)
pred_masks = (pred_masks > 0.35).astype("float32")

2026-03-07 14:51:33.856157: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 50331648 exceeds 10% of free system memory.
2026-03-07 14:51:33.968130: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 51121152 exceeds 10% of free system memory.
2026-03-07 14:51:34.649357: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 35651584 exceeds 10% of free system memory.
2026-03-07 14:51:34.862797: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 33554432 exceeds 10% of free system memory.
2026-03-07 14:51:34.925291: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 83886080 exceeds 10% of free system memory.


35/35 [==============================] - 68s 2s/step


In [17]:
y_true = []

for _, mask in test_dataset:
    y_true.append(mask.numpy())

y_true = np.concatenate(y_true, axis=0)

2026-03-07 14:53:29.824650: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [18]:
import numpy as np

def dice(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2 * intersection + 1e-7) / (np.sum(y_true) + np.sum(y_pred) + 1e-7)

def iou(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + 1e-7) / (union + 1e-7)

def precision(y_true, y_pred):
    tp = np.sum(y_true * y_pred)
    fp = np.sum((1 - y_true) * y_pred)
    return (tp + 1e-7) / (tp + fp + 1e-7)

def recall(y_true, y_pred):
    tp = np.sum(y_true * y_pred)
    fn = np.sum(y_true * (1 - y_pred))
    return (tp + 1e-7) / (tp + fn + 1e-7)

In [19]:
dice_scores = []
iou_scores = []
precision_scores = []
recall_scores = []

for i in range(len(pred_masks)):
    
    gt = y_true[i].flatten()
    pred = pred_masks[i].flatten()

    dice_scores.append(dice(gt, pred))
    iou_scores.append(iou(gt, pred))
    precision_scores.append(precision(gt, pred))
    recall_scores.append(recall(gt, pred))

print("Dice:", np.mean(dice_scores))
print("IoU:", np.mean(iou_scores))
print("Precision:", np.mean(precision_scores))
print("Recall:", np.mean(recall_scores))

Dice: 0.7348333436773684
IoU: 0.6391581806702791
Precision: 0.7624487349253174
Recall: 0.8496231225690233
